# Chip Dynamics — Batch Results Analysis

Load and explore CSV tables produced by **`batch_chip_dynamics.py`**.
Each CSV in the output directory represents one (CZI file × scene × channel) pair
and contains all detected objects across all timepoints.

Sections
--------
1. Configuration & Load
2. Summary table
3. Temporal dynamics — object count & area over time
4. Band occupancy — inside vs outside the microfluidic channel
5. Cell size distribution per timepoint
6. Spatial density heatmap
7. Export filtered results


## 1 · Configuration & Load

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

%load_ext autoreload
%autoreload 2
%matplotlib widget


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  USER PARAMETERS — edit this cell
# ════════════════════════════════════════════════════════════════════════════

# Directory that was passed as --output to batch_chip_dynamics.py
RESULTS_DIR = Path("/path/to/results")   # ← change

# Optional filters applied after loading (None = no filter)
FILTER_ORGANISM   = None   # e.g. "dictyostelium" or None
FILTER_CZI        = None   # substring match on czi_file name, or None

# Size / distance filters for all downstream plots
SEL_AREA_MIN_UM2 = 25.0    # minimum object area (µm²)
SEL_AREA_MAX_UM2 = None    # maximum object area  (None = no limit)
SEL_DIST_MAX_UM  = None    # max |distance to band centre| in µm (None = all)

# Colour palette: organism → matplotlib colour
ORG_COLORS = {
    "dictyostelium": "#1f77b4",
    "celegans":      "#2ca02c",
    "physarum":      "#d62728",
}


In [ ]:
# ── Load all *_objects.csv files in RESULTS_DIR ───────────────────────────────
csv_files = sorted(RESULTS_DIR.glob("*_objects.csv"))
print(f"Found {len(csv_files)} CSV file(s) in {RESULTS_DIR}")

if not csv_files:
    raise FileNotFoundError(
        f"No *_objects.csv files found in {RESULTS_DIR}.\n"
        "Check that RESULTS_DIR points to the --output directory of batch_chip_dynamics.py."
    )

dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    # back-compat: older runs may not have all metadata columns
    df["_source_file"] = f.name
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

# ── Normalise organism names ──────────────────────────────────────────────────
df_all["organism"] = df_all["organism"].str.strip().str.lower()

# ── Apply optional top-level filters ─────────────────────────────────────────
if FILTER_ORGANISM:
    df_all = df_all[df_all["organism"] == FILTER_ORGANISM.strip().lower()]
if FILTER_CZI:
    df_all = df_all[df_all["czi_file"].str.contains(FILTER_CZI, case=False)]

# ── Unique experiment key (CZI + scene + channel) ─────────────────────────────
df_all["experiment"] = (
    df_all["czi_file"].str.replace(r"\.czi$", "", regex=True)
    + "  s" + df_all["scene"].astype(str)
    + "  ch" + df_all["channel"].astype(str)
)

print(f"Total rows       : {len(df_all):,}")
print(f"Experiments      : {df_all['experiment'].nunique()}")
print(f"Organisms        : {df_all['organism'].unique().tolist()}")
print(f"Timepoint range  : {df_all['timepoint'].min()} – {df_all['timepoint'].max()}")
display(df_all.head(5))


## 2 · Summary Table

Per-experiment overview: total objects detected, timepoints covered, band geometry.

In [ ]:
summary = (
    df_all.groupby(["experiment", "organism", "czi_file", "scene", "channel"])
    .agg(
        n_objects      =("id",            "count"),
        n_timepoints   =("timepoint",     "nunique"),
        t_min          =("timepoint",     "min"),
        t_max          =("timepoint",     "max"),
        band_width_um  =("band_width_um", "first"),
        px_um          =("px_um",         "first"),
        rotation_deg   =("rotation_deg",  "first"),
        mean_area_um2  =("area_um2",      "mean"),
    )
    .reset_index()
    .sort_values("experiment")
)

print(f"Experiments: {len(summary)}")
display(
    summary.style
    .format({
        "band_width_um":  "{:.1f}",
        "px_um":          "{:.4f}",
        "rotation_deg":   "{:.2f}",
        "mean_area_um2":  "{:.1f}",
    })
    .background_gradient(subset=["n_objects"], cmap="YlOrRd")
)


## 3 · Temporal Dynamics — Object Count & Mean Area Over Time

Per-timepoint aggregate statistics, one line per experiment.
Dual y-axis: left = object count, right = mean area (µm²).

In [ ]:
# ── Apply size / distance filter ──────────────────────────────────────────────
def apply_filters(df):
    mask = pd.Series(True, index=df.index)
    if SEL_AREA_MIN_UM2 is not None:
        mask &= df["area_um2"] >= SEL_AREA_MIN_UM2
    if SEL_AREA_MAX_UM2 is not None:
        mask &= df["area_um2"] <= SEL_AREA_MAX_UM2
    if SEL_DIST_MAX_UM is not None:
        mask &= df["dist_to_band_um"].abs() <= SEL_DIST_MAX_UM
    return df[mask].copy()

df_sel = apply_filters(df_all)
print(f"After filters: {len(df_sel):,} objects  "
      f"({len(df_sel)/max(len(df_all),1)*100:.1f} % of total)")

# ── Per-timepoint stats ───────────────────────────────────────────────────────
tp_stats = (
    df_sel
    .groupby(["experiment", "organism", "timepoint"])
    .agg(
        count        =("id",       "count"),
        mean_area_um2=("area_um2", "mean"),
        total_area_um2=("area_um2","sum"),
    )
    .reset_index()
)

# ── Plot ──────────────────────────────────────────────────────────────────────
experiments = tp_stats["experiment"].unique()
# use a default palette if an organism colour is missing
def _color(org): return ORG_COLORS.get(org, "#888888")

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=False)
ax_count, ax_area = axes

for exp in experiments:
    sub  = tp_stats[tp_stats["experiment"] == exp].sort_values("timepoint")
    org  = sub["organism"].iloc[0]
    c    = _color(org)
    lbl  = f"{exp}  ({org})"
    ax_count.plot(sub["timepoint"], sub["count"],         color=c, lw=1.5, label=lbl)
    ax_area.plot( sub["timepoint"], sub["mean_area_um2"], color=c, lw=1.5, label=lbl, ls="--")

ax_count.set_ylabel("Object count per timepoint")
ax_count.set_xlabel("Timepoint")
ax_count.legend(fontsize=7, ncol=2, loc="upper right")
ax_count.set_title("Object count over time", fontsize=10)
ax_count.grid(True, alpha=0.25)

ax_area.set_ylabel("Mean area per timepoint (µm²)")
ax_area.set_xlabel("Timepoint")
ax_area.set_title("Mean object area over time", fontsize=10)
ax_area.grid(True, alpha=0.25)

plt.suptitle("Temporal dynamics", fontsize=11)
plt.tight_layout(); plt.show()


## 4 · Band Occupancy Analysis

Classify each object as **inside** the microfluidic channel band
(`|dist_to_band_um| ≤ band_half_width_um`) or **outside**.
Plot inside-vs-outside count over time and the occupancy ratio.

In [ ]:
# band_half_width_um is stored per-row; derive it from band_width_um
df_sel = df_sel.copy()
df_sel["band_half_width_um"] = df_sel["band_width_um"] / 2.0
df_sel["in_band"] = df_sel["dist_to_band_um"].abs() <= df_sel["band_half_width_um"]

occ = (
    df_sel
    .groupby(["experiment", "organism", "timepoint", "in_band"])
    .size()
    .rename("count")
    .reset_index()
)

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=False)
ax_cnt, ax_ratio = axes

for exp in df_sel["experiment"].unique():
    sub    = occ[occ["experiment"] == exp]
    org    = df_sel.loc[df_sel["experiment"] == exp, "organism"].iloc[0]
    c      = _color(org)
    inside = sub[sub["in_band"]].set_index("timepoint")["count"]
    outside= sub[~sub["in_band"]].set_index("timepoint")["count"]
    tp     = sorted(set(inside.index) | set(outside.index))
    inside  = inside.reindex(tp, fill_value=0)
    outside = outside.reindex(tp, fill_value=0)
    total   = inside + outside
    ratio   = inside / total.replace(0, np.nan)

    ax_cnt.plot(tp, inside,  color=c, lw=1.5, label=f"{exp} inside")
    ax_cnt.plot(tp, outside, color=c, lw=1.5, ls=":", alpha=0.6, label=f"{exp} outside")
    ax_ratio.plot(tp, ratio, color=c, lw=1.5, label=exp)

ax_cnt.set_ylabel("Object count")
ax_cnt.set_xlabel("Timepoint")
ax_cnt.set_title("Inside (solid) vs Outside (dotted) band", fontsize=10)
ax_cnt.legend(fontsize=7, ncol=2)
ax_cnt.grid(True, alpha=0.25)

ax_ratio.axhline(0.5, color="gray", lw=1, ls="--", alpha=0.5)
ax_ratio.set_ylim(0, 1)
ax_ratio.set_ylabel("Fraction inside band")
ax_ratio.set_xlabel("Timepoint")
ax_ratio.set_title("Band occupancy ratio", fontsize=10)
ax_ratio.legend(fontsize=7, ncol=2)
ax_ratio.grid(True, alpha=0.25)

plt.suptitle("Band occupancy", fontsize=11)
plt.tight_layout(); plt.show()


## 5 · Cell Size Distribution per Timepoint

Violin plots of `area_um2` grouped by experiment and binned time window (early / mid / late).
Objects outside the size filters are excluded.

In [ ]:
# ── Bin timepoints into three windows: early / mid / late ────────────────────
def _bin_timepoint(tp_series):
    lo, hi = tp_series.min(), tp_series.max()
    thirds = (hi - lo) / 3
    bins   = [lo - 1, lo + thirds, lo + 2 * thirds, hi + 1]
    labels = ["early", "mid", "late"]
    return pd.cut(tp_series, bins=bins, labels=labels)

df_violin = df_sel.copy()
df_violin["time_window"] = _bin_timepoint(df_violin["timepoint"])

# One subplot per experiment
experiments = df_violin["experiment"].unique()
n_exp = len(experiments)
fig, axes = plt.subplots(1, n_exp, figsize=(5 * n_exp, 5), sharey=True)
if n_exp == 1:
    axes = [axes]

for ax, exp in zip(axes, experiments):
    sub = df_violin[df_violin["experiment"] == exp]
    org = sub["organism"].iloc[0]
    c   = _color(org)
    windows = sub["time_window"].dropna().unique()

    data_by_window = [
        np.log10(sub.loc[sub["time_window"] == w, "area_um2"].values + 1)
        for w in ["early", "mid", "late"]
        if w in windows
    ]
    positions = [i for i, w in enumerate(["early", "mid", "late"]) if w in windows]
    labels    = [w for w in ["early", "mid", "late"] if w in windows]

    if data_by_window:
        parts = ax.violinplot(data_by_window, positions=positions, showmedians=True)
        for pc in parts["bodies"]:
            pc.set_facecolor(c); pc.set_alpha(0.55)
        parts["cmedians"].set_color("black")
    ax.set_xticks(positions); ax.set_xticklabels(labels)
    ax.set_title(exp, fontsize=8)
    ax.set_xlabel("Time window")

axes[0].set_ylabel("log₁₀ area (µm²)")
plt.suptitle("Cell area distribution — early / mid / late", fontsize=11)
plt.tight_layout(); plt.show()


## 6 · Spatial Density Heatmap

2D histogram of centroid positions across all selected timepoints,
normalised by the number of timepoints and overlaid on the reference brightfield image.

> **Note**: requires `img_bf_ref` — either load it here by pointing to a CZI,
> or set `SHOW_ON_IMAGE = False` to display the heatmap alone.

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
HEATMAP_EXPERIMENT = df_sel["experiment"].unique()[0]   # pick one experiment
HEATMAP_N_BINS     = 80        # spatial histogram bins (rows & cols)
SHOW_ON_IMAGE      = False     # True → overlay on BF image (needs CZI access)

# Optional: load BF reference image from the CZI for overlay
# (fill in CZI_PATH, SCENE, CHANNEL_BF if SHOW_ON_IMAGE = True)
CZI_PATH_HEATMAP  = None   # e.g. "/path/to/file.czi"
SCENE_HEATMAP     = 0
CHANNEL_BF_HEATMAP = 2

# ── Build heatmap ─────────────────────────────────────────────────────────────
sub = df_sel[df_sel["experiment"] == HEATMAP_EXPERIMENT]

if sub.empty:
    print("No data for selected experiment.")
else:
    row_vals = sub["centroid_row"].values
    col_vals = sub["centroid_col"].values
    n_tp     = sub["timepoint"].nunique()

    # Auto-detect image bounds from data
    row_max = int(np.ceil(row_vals.max())) + 1
    col_max = int(np.ceil(col_vals.max())) + 1

    H2d, row_edges, col_edges = np.histogram2d(
        row_vals, col_vals,
        bins=HEATMAP_N_BINS,
        range=[[0, row_max], [0, col_max]],
    )
    H2d_norm = H2d / n_tp   # normalise by number of timepoints

    # ── Plot ──────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 6))

    if SHOW_ON_IMAGE and CZI_PATH_HEATMAP is not None:
        from chipanalysis.functions.chip_dynamics import load_czi, get_rotation_fn, get_rotated_frame
        _czi, _px, _, _ = load_czi(CZI_PATH_HEATMAP)
        _rfn, _ = get_rotation_fn(_czi, bf_channel=CHANNEL_BF_HEATMAP,
                                  scene=SCENE_HEATMAP, px_um=_px)
        _img, _ = get_rotated_frame(_czi, 0, CHANNEL_BF_HEATMAP, SCENE_HEATMAP, _rfn)
        ax.imshow(_img, cmap="gray", origin="upper",
                  extent=[0, col_max, row_max, 0], zorder=0)
        cmap_heat = "hot"
        alpha_heat = 0.65
    else:
        cmap_heat = "viridis"
        alpha_heat = 1.0

    extent = [col_edges[0], col_edges[-1], row_edges[-1], row_edges[0]]
    im = ax.imshow(H2d_norm, origin="upper", extent=extent,
                   cmap=cmap_heat, alpha=alpha_heat, zorder=1)
    plt.colorbar(im, ax=ax, label="Mean detections per timepoint")

    # Band boundaries from first row of this experiment
    row0 = sub.iloc[0]
    if "band_top_px" in row0.index:
        ax.axhline(row0["band_top_px"],    color="cyan", lw=1.2, ls="--", label="band top")
        ax.axhline(row0["band_bottom_px"], color="cyan", lw=1.2, ls="--", label="band bottom")
        ax.legend(fontsize=8)

    ax.set_xlabel("Column (px)")
    ax.set_ylabel("Row (px)")
    ax.set_title(
        f"Spatial density heatmap — {HEATMAP_EXPERIMENT}\n"
        f"({n_tp} timepoints, {len(sub):,} objects)",
        fontsize=10,
    )
    plt.tight_layout(); plt.show()


## 7 · Export Filtered Results

Save `df_sel` (filtered objects) and the per-experiment summary to CSV.
All outputs land in `RESULTS_DIR/analysis/` with a timestamp prefix.

In [ ]:
import datetime

timestamp   = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
export_dir  = RESULTS_DIR / "analysis"
export_dir.mkdir(parents=True, exist_ok=True)

# ── Filtered objects table ────────────────────────────────────────────────────
out_sel = export_dir / f"{timestamp}_objects_filtered.csv"
df_sel.drop(columns=["_source_file", "band_half_width_um", "in_band"],
            errors="ignore").to_csv(out_sel, index=False)
print(f"Saved filtered objects → {out_sel.name}  ({len(df_sel):,} rows)")

# ── Summary table ─────────────────────────────────────────────────────────────
out_sum = export_dir / f"{timestamp}_summary.csv"
summary.to_csv(out_sum, index=False)
print(f"Saved summary          → {out_sum.name}  ({len(summary)} experiments)")

# ── Temporal stats ────────────────────────────────────────────────────────────
out_tp  = export_dir / f"{timestamp}_temporal_stats.csv"
tp_stats.to_csv(out_tp, index=False)
print(f"Saved temporal stats   → {out_tp.name}")

# ── Band occupancy ────────────────────────────────────────────────────────────
out_occ = export_dir / f"{timestamp}_band_occupancy.csv"
occ.to_csv(out_occ, index=False)
print(f"Saved band occupancy   → {out_occ.name}")

print(f"\nAll exports in: {export_dir}")
